In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report,confusion_matrix

In [ ]:
data = pd.read_csv("data/diabetes_data.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'diabetes_data.csv'

In [ ]:
data.head(10)

In [ ]:
data.columns.str.lower().str.replace(' ', '_')

In [ ]:
data.columns = data.columns.str.lower().str.replace(' ', '_')

In [ ]:
data.head(5)

In [ ]:
data.info()

In [ ]:
data.isna().sum()

In [ ]:
data.describe()

In [ ]:
data.dtypes

In [ ]:
data.count()

In [ ]:
data.shape

In [ ]:
data['label'].value_counts()

#### Narrative
1 = Positive
0 = Negative

in label which is our target.

In [ ]:
target_label = {'Positive':1, 'Negative':0}

In [ ]:
#Plotting count values of Positive and Negative

data['label'].value_counts().plot(kind = 'bar')

In [ ]:
data['gender'].value_counts()

#### Narrative
1 = Male
0 = Female

in gender.

In [ ]:
data['gender'].value_counts().plot(kind = 'bar')

In [ ]:
data.groupby(['age', 'gender']).size()

### freuqency distribution table using the age range




In [ ]:
data['age'].max()

In [ ]:
data['age'].min()

In [ ]:
labels = ["Less than 20", "20 to 30", "30 to 40", "40 to 50", "50 to 60",
         "60 to 70", "70 to 80", "80 to 90"]

bins = [10, 20, 30, 40, 50, 60, 70, 80, 90]

freq_data = data.groupby(pd.cut(data['age'], bins=bins, labels=labels)).size()

In [ ]:
freq_data

In [ ]:
freq_data = freq_data.reset_index(name='count')

In [ ]:
freq_data

In [ ]:
freq_data.plot(kind = 'bar')

In [ ]:
freq_data.plot(kind = 'line')

In [ ]:
plt.bar(freq_data['age'], freq_data['count'])
plt.ylabel('Counts')
plt.title('Frequency Counts of Age')
plt.show()

In [ ]:
#check and detect outliers

+ univariate and multivariate analysis
+ using box plot for univariate
+ using scatterplot for multivariate
+ using z-score
+ using IQT

In [ ]:
#using bocplot for univariate
sns.boxplot(data['age'])

#### we can see couple of outliers in 80-90 age range

In [ ]:
data[data['age'] == 90]

In [ ]:
#scatter for multi vraiables outlier detection

In [ ]:
sns.scatterplot(data['age'], data['delayed_healing'])

### Feature Engineering

+ SelectKbest: to fin out the best features having the strong relationship with our target/output
+ Recursive feature elimination
+ Ranking best mechanism

In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

In [ ]:
#features and labels

In [ ]:
data.columns

In [ ]:
xfeatures = data[['age', 'gender', 'polyuria', 'polydipsia', 'sudden_weight_loss',
       'weakness', 'polyphagia', 'genital_thrush', 'visual_blurring',
       'itching', 'irritability', 'delayed_healing', 'partial_paresis',
       'muscle_stiffness', 'alopecia', 'obesity']]

ylabels = data['label']

In [ ]:
#using skbest
skb = SelectKBest(score_func=chi2, k=12)
best_feature = skb.fit(xfeatures,ylabels)

In [ ]:
#highest score meaning better features
print("Score", best_feature.scores_)

In [ ]:
best_feature_fit = best_feature.transform(xfeatures)

In [ ]:
best_feature_fit

In [ ]:
#mapping to features name
feature_score = pd.DataFrame(best_feature.scores_, columns=['Feature Scores'])

In [ ]:
feature_score

In [ ]:
feature_column_names = pd.DataFrame(xfeatures.columns, columns=['feature_name'])
best_feature_dataframe = pd.concat([feature_score, feature_column_names], axis =1)

In [ ]:
best_feature_dataframe

In [ ]:
#get the best among them from higher scores

In [ ]:
best_feature_dataframe.nlargest(12, 'Feature Scores')

### checking for feature importance
+ extra tree classifier

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

In [ ]:
et_clf = ExtraTreesClassifier()
et_clf.fit(xfeatures, ylabels)

In [ ]:
#print important 
print(et_clf.feature_importances_)

In [ ]:
feature_importance_dataframe = pd.Series(et_clf.feature_importances_, index=xfeatures.columns)

In [ ]:
feature_importance_dataframe

In [ ]:
feature_importance_dataframe.nlargest(12).plot(kind = 'barh')

In [ ]:
data.corr()

In [ ]:
plt.figure(figsize=(20,10))
sns.heatmap(xfeatures.corr(), annot=True)
plt.show()

## Building Model

+ Logistic Regression
+ Random Forests
+ Decision Tree

In [ ]:
xfeatures

In [ ]:
ylabels

In [ ]:
# for selected best features
best_feature_dataframe.nlargest(12, 'Feature Scores')

In [ ]:
type(best_feature_dataframe.nlargest(12, 'Feature Scores'))

In [ ]:
best_feature_dataframe.nlargest(12, 'Feature Scores')['feature_name']

In [ ]:
best_feature_dataframe.nlargest(12, 'Feature Scores')['feature_name'].unique()

In [ ]:
data.columns

In [ ]:
xfeatures_best = data[['age', 'gender', 'polyuria', 'polydipsia', 'sudden_weight_loss','weakness', 'polyphagia', 'genital_thrush', 'visual_blurring','irritability','partial_paresis','alopecia']]

In [ ]:
xfeatures_best

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(xfeatures_best, ylabels, test_size=0.25, random_state=42)

In [ ]:
logreg = LogisticRegression()
model_logit = logreg.fit(x_train, y_train)


In [ ]:
model_logit.score(x_test, y_test)

In [ ]:
#method 2
model_logit.predict(x_test)

In [ ]:
print(list(y_test))

In [ ]:
accuracy_score(y_test, model_logit.predict(x_test))

In [ ]:
x_test.iloc[1]

In [ ]:
result_dic = {"Positive": 1, "Negative": 0}
x1 = x_test.iloc[1]

In [ ]:
model_logit.predict(np.array(x1).reshape(1, -1))

## Wow, correct so far. Looks good!!

In [ ]:
#Decision Tree

In [ ]:
clf = DecisionTreeClassifier()
model_tree = clf.fit(x_train, y_train)

In [ ]:
model_tree.score(x_test, y_test)

In [ ]:
#KNN Algorithm

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn = KNeighborsClassifier(n_neighbors=4)

In [ ]:
model_knn = knn.fit(x_train, y_train)

In [ ]:
model_knn.score(x_test, y_test)

### save our models 
+ pickle
+ joblib
+ json etc

In [ ]:
#using joblib
import joblib

In [ ]:
model_file = open("logistic_regression_model.pkl", "wb")

In [ ]:
joblib.dump(model_logit, model_file)
model_file.close()

In [ ]:
model_file_tree = open("decision_tree_model.pkl", "wb")
joblib.dump(model_tree, model_file_tree)
model_file_tree.close()

In [ ]:
model_file_knn = open("knn_model.pkl", "wb")
joblib.dump(model_knn, model_file_knn)
model_file_knn.close()

## Create decision tree plot

In [ ]:
from IPython.display import Image
from sklearn import tree
import pydotplus

In [ ]:
xfeatures_best.columns

In [ ]:
features_name_best = xfeatures_best.columns

In [ ]:
target_names = ["Positive", "Negative"]

In [ ]:
dot_data = tree.export_graphviz(clf, out_file=None, feature_names=features_name_best, class_names=target_names)

In [ ]:
graph = pydotplus.graph_from_dot_data(dot_data)

In [ ]:
Image(graph.create_png())

In [ ]:
#save the image
graph.write_png("decision_tree_plot.png")

## Interpret and Evaluate
+ Eli5
+ Lime
+ SHAP
+ Skater

In [ ]:
#using Lime for Interpreting
import lime
import lime.lime_tabular

In [ ]:
dir(lime)

In [ ]:
#create lime explainer using table, testexplainer, image explainer


In [ ]:
features_name_best

In [ ]:
target_names

In [ ]:
class_names = ["Negative(0)", "Positive(1)"]

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(x_train.values,
                                                   feature_names=features_name_best,
                                                   class_names=class_names,discretize_continuous=True)

In [ ]:
x1

In [ ]:
x_test.iloc[1]

In [ ]:
x_test.iloc[54]

In [ ]:
model_tree.predict(np.array(x_test.iloc[54]).reshape(1,-1))

In [ ]:
model_tree.predict(np.array(x_test.iloc[1]).reshape(1,-1))

In [ ]:
exp = explainer.explain_instance(x_test.iloc[54], model_tree.predict_proba, num_features=12, top_labels=1)

In [ ]:
exp.show_in_notebook(show_table = True, show_all = False)

## Evaluation

In [ ]:
y_pred = model_tree.predict(x_test)

In [ ]:
accuracy_score(y_test, y_pred)

In [ ]:
#confusion matrix
confusion_matrix(y_test, y_pred)

In [ ]:
#plot
from sklearn.metrics import plot_confusion_matrix

In [ ]:
plot_confusion_matrix(model_tree,x_test,y_test)

In [ ]:
print(classification_report(y_test, y_pred, target_names=class_names))